# 3. Agentify and vector search

This notebook prepares markdown outputs for vector search and creates a vector search index.

Inputs:
- Markdown files in the processed Volume

Outputs:
- Delta table with markdown content
- Vector search index

In [ ]:
%pip install uv
%sh uv pip install .
%sh uv pip install ".[local]"
%restart_python

In [ ]:
from pyspark.sql.functions import col, input_file_name, sha2
import mlflow
import os
from pathlib import Path

from utils import get_volume_path, is_local_env, resolve_output_root

config = mlflow.models.ModelConfig(development_config="./config.yaml")
config = config.to_dict()

if is_local_env():
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()

CATALOG = config["catalog"]
SCHEMA = config["schema"]
PROCESSED_VOLUME = config["output_volume"]
TABLE_NAME = config["parsed_markdown_table"]

local_output_path = config.get("local_output_path")
is_local = is_local_env()

source_table = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

processed_root = resolve_output_root(CATALOG, SCHEMA, PROCESSED_VOLUME, local_output_path)

if is_local:
    markdown_files = sorted(processed_root.rglob("doc.md"))
    rows = [
        {"doc_path": str(path), "content": path.read_text(encoding="utf-8")}
        for path in markdown_files
    ]
    df = spark.createDataFrame(rows)
else:
    markdown_glob = f"{processed_root}/*/doc.md"
    df = (
        spark.read.text(markdown_glob)
        .withColumnRenamed("value", "content")
        .withColumn("doc_path", input_file_name())
    )

df = df.withColumn("id", sha2(col("doc_path"), 256))

df.write.mode("overwrite").saveAsTable(source_table)
print(f"Created table: {source_table}")

In [ ]:
from databricks.vector_search.client import VectorSearchClient

VECTOR_SEARCH_ENDPOINT = config["vector_search_endpoint"]
INDEX_NAME = config["index_name"]
EMBEDDING_ENDPOINT = config["embedding_endpoint"]

index_fullname = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"

vsc = VectorSearchClient()

try:
    vsc.get_endpoint(name=VECTOR_SEARCH_ENDPOINT)
except Exception:
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT, endpoint_type="STANDARD")

vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    source_table_name=source_table,
    index_name=index_fullname,
    pipeline_type=config["index_pipeline_type"],
    primary_key="id",
    embedding_source_column="content",
    embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
)

print(f"Vector search index ready: {index_fullname}")

In [ ]:
index = vsc.get_index(endpoint_name=VECTOR_SEARCH_ENDPOINT, index_name=index_fullname)
print(index.describe())